# Step Counter - CNN Data Preparation

This notebook prepares raw windowed accelerometer data for 1D CNN training.

## Approach
1. Window the continuous signal into fixed-size segments
2. Store raw accelerometer values (no feature extraction)
3. Create multiple label types (binary, count, per-sample)
4. Split by participant to prevent data leakage
5. Save as numpy arrays for efficient CNN training

## CNN Input Format
- **Shape**: (n_windows, window_size, n_channels)
- **Example**: (136000, 200, 3) for 200 samples × 3 axes (x, y, z)
- **Optional**: Add 4th channel for SMV or ENMO

In [ ]:
# Import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [ ]:
# Load data paths

data_path = Path('../data/OxWalk_Dec2022/Hip_100Hz')
metadata_path = Path('../data/OxWalk_Dec2022/metadata.csv')

# Load metadata
metadata = pd.read_csv(metadata_path)
print(f'Total participants: {len(metadata)}')
print(metadata.head())

In [ ]:
# Configuration

SAMPLING_RATE = 100  # Hz
WINDOW_SIZE_SEC = 2.0  # seconds
WINDOW_SIZE = int(WINDOW_SIZE_SEC * SAMPLING_RATE)  # samples
OVERLAP = 0.5  # 50% overlap
STEP_SIZE = int(WINDOW_SIZE * (1 - OVERLAP))

# Channel configuration
USE_SMV_CHANNEL = True  # Add SMV as 4th channel
USE_ENMO_CHANNEL = False  # Add ENMO as additional channel

print(f'Window size: {WINDOW_SIZE} samples ({WINDOW_SIZE_SEC} seconds)')
print(f'Step size: {STEP_SIZE} samples ({STEP_SIZE/SAMPLING_RATE} seconds)')
print(f'Overlap: {OVERLAP*100}%')
print(f'Add SMV channel: {USE_SMV_CHANNEL}')
print(f'Add ENMO channel: {USE_ENMO_CHANNEL}')

In [ ]:
# Window extraction function

def process_participant_cnn(participant_id, window_size=WINDOW_SIZE, step_size=STEP_SIZE,
                            use_smv=USE_SMV_CHANNEL, use_enmo=USE_ENMO_CHANNEL):
    """
    Process a single participant's data and extract raw windowed data for CNN.
    
    Returns:
        windows: numpy array of shape (n_windows, window_size, n_channels)
        labels_dict: dictionary with different label types
        metadata_list: list of metadata dicts for each window
    """
    
    # Load data
    file_path = data_path / f'{participant_id}_hip100.csv'
    df = pd.read_csv(file_path)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    # Calculate derived channels if needed
    if use_smv or use_enmo:
        df['smv'] = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2)
    if use_enmo:
        df['enmo'] = np.maximum(df['smv'] - 1, 0)
    
    windows_list = []
    labels_binary = []
    labels_count = []
    labels_per_sample = []
    metadata_list = []
    
    # Sliding window
    for start_idx in range(0, len(df) - window_size + 1, step_size):
        end_idx = start_idx + window_size
        window = df.iloc[start_idx:end_idx]
        
        # Extract raw accelerometer data
        channels = []
        channels.append(window['x'].values)
        channels.append(window['y'].values)
        channels.append(window['z'].values)
        
        if use_smv:
            channels.append(window['smv'].values)
        if use_enmo:
            channels.append(window['enmo'].values)
        
        # Stack channels: (window_size, n_channels)
        window_data = np.stack(channels, axis=-1)
        windows_list.append(window_data)
        
        # Create labels
        annotations = window['annotation'].values
        step_count = annotations.sum()
        
        labels_binary.append(int(step_count > 0))  # Binary: has steps?
        labels_count.append(step_count)  # Count: how many steps?
        labels_per_sample.append(annotations)  # Per-sample: exact locations
        
        # Store metadata
        metadata_list.append({
            'participant': participant_id,
            'window_start': start_idx,
            'window_end': end_idx,
            'timestamp_start': window['timestamp'].iloc[0],
            'timestamp_end': window['timestamp'].iloc[-1]
        })
    
    # Convert to numpy arrays
    windows = np.array(windows_list)  # (n_windows, window_size, n_channels)
    
    labels_dict = {
        'binary': np.array(labels_binary),  # (n_windows,)
        'count': np.array(labels_count),  # (n_windows,)
        'per_sample': np.array(labels_per_sample)  # (n_windows, window_size)
    }
    
    return windows, labels_dict, metadata_list


print('Window extraction function defined')

In [ ]:
# Test on single participant

print('Processing P01...')
test_windows, test_labels, test_metadata = process_participant_cnn('P01')

print(f'\nWindows shape: {test_windows.shape}')
print(f'  - Number of windows: {test_windows.shape[0]}')
print(f'  - Window size (timesteps): {test_windows.shape[1]}')
print(f'  - Number of channels: {test_windows.shape[2]}')

print(f'\nLabels:')
print(f'  - Binary shape: {test_labels["binary"].shape}')
print(f'  - Count shape: {test_labels["count"].shape}')
print(f'  - Per-sample shape: {test_labels["per_sample"].shape}')

print(f'\nLabel distributions:')
print(f'  - Binary (has_steps): {np.bincount(test_labels["binary"])}')
print(f'  - Count statistics:')
print(f'    Mean: {test_labels["count"].mean():.2f}')
print(f'    Std: {test_labels["count"].std():.2f}')
print(f'    Max: {test_labels["count"].max()}')
print(f'    Total steps: {test_labels["count"].sum()}')

print(f'\nFirst window data sample:')
print(f'Shape: {test_windows[0].shape}')
print(f'First 5 timesteps:')
print(test_windows[0][:5])

In [ ]:
# Visualize a sample window with steps

# Find a window with steps
step_window_idx = np.where(test_labels['binary'] == 1)[0][10]  # 10th window with steps

sample_window = test_windows[step_window_idx]
sample_annotations = test_labels['per_sample'][step_window_idx]

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

time = np.arange(WINDOW_SIZE) / SAMPLING_RATE

# Plot each channel
axes[0].plot(time, sample_window[:, 0], label='X (Superior)', linewidth=0.8)
axes[0].set_ylabel('X (g)')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

axes[1].plot(time, sample_window[:, 1], label='Y (Anterior)', color='orange', linewidth=0.8)
axes[1].set_ylabel('Y (g)')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

axes[2].plot(time, sample_window[:, 2], label='Z (Lateral)', color='green', linewidth=0.8)
axes[2].set_ylabel('Z (g)')
axes[2].legend(loc='upper right')
axes[2].grid(True, alpha=0.3)

if USE_SMV_CHANNEL:
    axes[3].plot(time, sample_window[:, 3], label='SMV', color='purple', linewidth=0.8)
    axes[3].set_ylabel('SMV (g)')
    axes[3].legend(loc='upper right')
    axes[3].grid(True, alpha=0.3)

# Mark step annotations
step_times = time[sample_annotations == 1]
for ax in axes:
    for step_time in step_times:
        ax.axvline(step_time, color='red', linestyle='--', alpha=0.5, linewidth=1)

axes[3].set_xlabel('Time (seconds)')
plt.suptitle(f'Sample Window with {sample_annotations.sum()} Steps (Window {step_window_idx})', fontsize=14)
plt.tight_layout()
plt.show()

print(f'Window {step_window_idx}: {sample_annotations.sum()} steps')

In [ ]:
# Process all participants

all_windows = []
all_labels_binary = []
all_labels_count = []
all_labels_per_sample = []
all_metadata = []
all_participants = []

for idx, row in metadata.iterrows():
    pid = row['participant']
    try:
        print(f'Processing {pid}...', end=' ')
        windows, labels, metadata_list = process_participant_cnn(pid)
        
        all_windows.append(windows)
        all_labels_binary.append(labels['binary'])
        all_labels_count.append(labels['count'])
        all_labels_per_sample.append(labels['per_sample'])
        
        # Add demographic info to metadata
        for meta in metadata_list:
            meta['sex'] = row['sex']
            meta['age'] = row['age']
        all_metadata.extend(metadata_list)
        
        # Track participant for each window
        all_participants.extend([pid] * len(windows))
        
        print(f'{len(windows)} windows extracted')
    except Exception as e:
        print(f'Error: {e}')

# Combine all participants
X = np.concatenate(all_windows, axis=0)
y_binary = np.concatenate(all_labels_binary, axis=0)
y_count = np.concatenate(all_labels_count, axis=0)
y_per_sample = np.concatenate(all_labels_per_sample, axis=0)
participants_array = np.array(all_participants)

print(f'\n=== Dataset Summary ===')
print(f'Total windows: {len(X)}')
print(f'X shape: {X.shape}')
print(f'y_binary shape: {y_binary.shape}')
print(f'y_count shape: {y_count.shape}')
print(f'y_per_sample shape: {y_per_sample.shape}')
print(f'\nClass distribution (binary):')
unique, counts = np.unique(y_binary, return_counts=True)
for val, count in zip(unique, counts):
    print(f'  {val}: {count} ({count/len(y_binary)*100:.1f}%)')
print(f'\nStep count distribution:')
print(f'  Mean: {y_count.mean():.2f}')
print(f'  Std: {y_count.std():.2f}')
print(f'  Min: {y_count.min()}')
print(f'  Max: {y_count.max()}')
print(f'  Total steps: {y_count.sum()}')

In [ ]:
# Visualize data statistics

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Binary class distribution
axes[0, 0].bar(['No steps', 'Has steps'], np.bincount(y_binary), edgecolor='black', alpha=0.7)
axes[0, 0].set_ylabel('Number of Windows')
axes[0, 0].set_title('Binary Classification: Step Presence')
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Step count distribution
axes[0, 1].hist(y_count, bins=range(int(y_count.max())+2), edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Number of Steps per Window')
axes[0, 1].set_ylabel('Number of Windows')
axes[0, 1].set_title('Step Count Distribution')
axes[0, 1].grid(True, alpha=0.3)

# Accelerometer value distribution (all windows, X axis)
axes[1, 0].hist(X[:, :, 0].flatten(), bins=50, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Acceleration (g)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('X-axis Acceleration Distribution')
axes[1, 0].grid(True, alpha=0.3)

# SMV distribution if available
if USE_SMV_CHANNEL:
    axes[1, 1].hist(X[:, :, 3].flatten(), bins=50, edgecolor='black', alpha=0.7, color='purple')
    axes[1, 1].set_xlabel('SMV (g)')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Signal Magnitude Vector Distribution')
    axes[1, 1].grid(True, alpha=0.3)
else:
    axes[1, 1].hist(X[:, :, 1].flatten(), bins=50, edgecolor='black', alpha=0.7, color='orange')
    axes[1, 1].set_xlabel('Acceleration (g)')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Y-axis Acceleration Distribution')
    axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Train/test split by participant

unique_participants = metadata['participant'].unique()
train_pids, test_pids = train_test_split(unique_participants, test_size=0.2, random_state=42)

print(f'Total participants: {len(unique_participants)}')
print(f'Train participants: {len(train_pids)} ({len(train_pids)/len(unique_participants)*100:.1f}%)')
print(f'Test participants: {len(test_pids)} ({len(test_pids)/len(unique_participants)*100:.1f}%)')
print(f'\nTest participants: {sorted(test_pids)}')

# Create train/test masks
train_mask = np.isin(participants_array, train_pids)
test_mask = np.isin(participants_array, test_pids)

# Split data
X_train = X[train_mask]
X_test = X[test_mask]

y_train_binary = y_binary[train_mask]
y_test_binary = y_binary[test_mask]

y_train_count = y_count[train_mask]
y_test_count = y_count[test_mask]

y_train_per_sample = y_per_sample[train_mask]
y_test_per_sample = y_per_sample[test_mask]

print(f'\n=== Train Set ===')
print(f'X_train shape: {X_train.shape}')
print(f'Class balance (binary): {y_train_binary.mean()*100:.1f}% with steps')
print(f'Total steps: {y_train_count.sum()}')

print(f'\n=== Test Set ===')
print(f'X_test shape: {X_test.shape}')
print(f'Class balance (binary): {y_test_binary.mean()*100:.1f}% with steps')
print(f'Total steps: {y_test_count.sum()}')

In [ ]:
# Data normalization (important for CNN training)

# Calculate statistics from training data only
train_mean = X_train.mean(axis=(0, 1))  # Mean per channel
train_std = X_train.std(axis=(0, 1))    # Std per channel

print('Training data statistics (per channel):')
channel_names = ['X', 'Y', 'Z']
if USE_SMV_CHANNEL:
    channel_names.append('SMV')
if USE_ENMO_CHANNEL:
    channel_names.append('ENMO')

for i, name in enumerate(channel_names):
    print(f'  {name}: mean={train_mean[i]:.4f}, std={train_std[i]:.4f}')

# Normalize (standardize) the data
X_train_norm = (X_train - train_mean) / train_std
X_test_norm = (X_test - train_mean) / train_std

print(f'\nNormalized train data: mean={X_train_norm.mean():.4f}, std={X_train_norm.std():.4f}')
print(f'Normalized test data: mean={X_test_norm.mean():.4f}, std={X_test_norm.std():.4f}')

In [ ]:
# Save processed data

output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

# Save as compressed numpy format (.npz)
print('Saving training data...')
np.savez_compressed(
    output_dir / 'cnn_train_data.npz',
    X=X_train_norm,
    y_binary=y_train_binary,
    y_count=y_train_count,
    y_per_sample=y_train_per_sample,
    participants=participants_array[train_mask]
)
print(f'Saved: {output_dir / "cnn_train_data.npz"}')

print('Saving test data...')
np.savez_compressed(
    output_dir / 'cnn_test_data.npz',
    X=X_test_norm,
    y_binary=y_test_binary,
    y_count=y_test_count,
    y_per_sample=y_test_per_sample,
    participants=participants_array[test_mask]
)
print(f'Saved: {output_dir / "cnn_test_data.npz"}')

# Save normalization parameters
print('Saving normalization parameters...')
np.savez(
    output_dir / 'cnn_normalization.npz',
    mean=train_mean,
    std=train_std,
    channel_names=channel_names
)
print(f'Saved: {output_dir / "cnn_normalization.npz"}')

# Save participant splits
pd.DataFrame({'participant': train_pids, 'split': 'train'}).to_csv(
    output_dir / 'cnn_train_participants.csv', index=False
)
pd.DataFrame({'participant': test_pids, 'split': 'test'}).to_csv(
    output_dir / 'cnn_test_participants.csv', index=False
)
print(f'Saved participant splits')

# Save metadata
pd.DataFrame(all_metadata).to_csv(output_dir / 'cnn_window_metadata.csv', index=False)
print(f'Saved window metadata')

print('\n=== Data Preparation Complete! ===')
print(f'\nFiles saved to: {output_dir.absolute()}')
print(f'\nTo load the data in your CNN training script:')
print(f'```python')
print(f'train_data = np.load("../data/processed/cnn_train_data.npz")')
print(f'X_train = train_data["X"]')
print(f'y_train = train_data["y_binary"]  # or y_count, y_per_sample')
print(f'```')

## Summary

This notebook prepared raw windowed data for CNN training:

### Data Format:
- **Input shape**: (n_windows, 200, 3) or (n_windows, 200, 4) with SMV
- **Window size**: 2 seconds (200 samples at 100 Hz)
- **Overlap**: 50%
- **Channels**: X, Y, Z (+ optional SMV/ENMO)

### Labels:
Three label types provided for flexibility:
1. **Binary** (`y_binary`): Has steps (0/1) - for binary classification
2. **Count** (`y_count`): Number of steps (0, 1, 2, ...) - for regression or multi-class
3. **Per-sample** (`y_per_sample`): Step location per timestep (200,) - for sequence labeling

### Preprocessing:
- Data normalized (standardized) per channel using training statistics
- Train/test split by participant (80/20) to prevent data leakage
- Saved as compressed numpy arrays for efficient loading

### Recommended Label Strategy:
Start with **count regression** (`y_count`) as it provides more information than binary classification:
- Loss: MSE or Huber loss
- Can threshold predictions for binary classification
- Can sum predictions to get total step count

### Next Steps:
1. Build 1D CNN architecture
2. Train model with binary or count labels
3. Evaluate on held-out test participants
4. Compare with traditional ML baselines